In [1]:
import networkx as nx
from sentence_transformers import SentenceTransformer
import numpy as np
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

# Step 1: Initialize Gemini API and LLM
import config

/Users/nitastha/Desktop/NitishFiles/Projects/SteamApps/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:

# Set up Gemini API
model_name = config.CHAT_MODEL  # Gemini Pro model
google_api_key = config.GOOGLE_API_KEY  # Replace with your actual Google API key

llm = ChatGoogleGenerativeAI(
    model=model_name,
    google_api_key=google_api_key,
    temperature=0.7,
    max_output_tokens=100
)

system_prompt = (
    """<s>[INST] You are a helpful assistant. Kindly answer the question based on the context provided. Do not give any additional commentry. If answer is not present say: Not found.

संदर्भ: {context} </s>
"""
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)


In [5]:

# Step 2: Initialize SentenceTransformer for Embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")

# Step 3: Chunk the Document
def chunk_document(text, chunk_size=300, overlap=50):
    tokens = text.split()
    chunks = []
    for i in range(0, len(tokens), chunk_size - overlap):
        chunk = " ".join(tokens[i:i+chunk_size])
        chunks.append(chunk)
    return chunks

# Step 4: Add Document Chunks as Nodes
def add_document_to_graph(doc_id, text, graph, chunk_size=300, overlap=50):
    chunks = chunk_document(text, chunk_size, overlap)
    embeddings = model.encode(chunks)
    for i, chunk in enumerate(chunks):
        node_id = f"{doc_id}_chunk_{i}"
        graph.add_node(node_id, text=chunk, embedding=embeddings[i])
        
        # Optionally, add edges between consecutive chunks
        if i > 0:
            graph.add_edge(f"{doc_id}_chunk_{i-1}", node_id, weight=1.0)
    return graph

# Step 5: Add Relationships Between Nodes
def add_semantic_edges(graph, threshold=0.8):
    nodes = list(graph.nodes(data=True))
    embeddings = [data["embedding"] for _, data in nodes]
    for i, emb1 in enumerate(embeddings):
        for j, emb2 in enumerate(embeddings):
            if i != j:
                similarity = np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))
                if similarity > threshold:
                    graph.add_edge(nodes[i][0], nodes[j][0], weight=similarity)

# Step 6: Query the Graph
def retrieve_relevant_chunks(graph, query, top_k=5):
    query_embedding = model.encode([query])[0]
    nodes = list(graph.nodes(data=True))
    similarities = [
        (node, np.dot(query_embedding, data["embedding"]) / (np.linalg.norm(query_embedding) * np.linalg.norm(data["embedding"])))
        for node, data in nodes
    ]
    # Sort nodes by similarity
    similarities = sorted(similarities, key=lambda x: x[1], reverse=True)[:top_k]
    return [(node, graph.nodes[node]["text"]) for node, _ in similarities]

def query_with_gemini(relevant_chunks, query):
    context = "\n".join(relevant_chunks)
    response = llm(prompt.format_messages(context=context, input=query))
    return response.content # Access the content attribute of the AIMessage object




In [12]:
def retrieve_relevant_chunks(graph, query, top_k=5, similarity_threshold=0.85):
    query_embedding = model.encode([query])[0]
    nodes = list(graph.nodes(data=True))
    similarities = [
        (node, np.dot(query_embedding, data["embedding"]) / (np.linalg.norm(query_embedding) * np.linalg.norm(data["embedding"])))
        for node, data in nodes
    ]
    # Filter by similarity threshold and sort
    similarities = [(node, score) for node, score in similarities if score > similarity_threshold]
    similarities = sorted(similarities, key=lambda x: x[1], reverse=True)[:top_k]
    return [(node, graph.nodes[node]["text"]) for node, _ in similarities]


def query_with_gemini(relevant_chunks, query):
    context = "\n---\n".join(relevant_chunks)  # Add clear chunk separators
    response = llm(prompt.format_messages(context=context, input=query))
    return response.content


In [14]:
# Example Usage
if __name__ == "__main__":
    # Initialize the graph
    graph = nx.Graph()
    
    # Add documents
    doc1 = "This is the first document. It contains multiple sentences and is fairly short."
    # doc2 = "This document discusses the concept of Graph RAG and how to build it."
    doc2 = "Graph RAG stands for Graph-based Retrieval-Augmented Generation. It is a system that uses a graph structure to represent document chunks and their relationships. It helps retrieve relevant information based on a query and provides it to a generative model to generate precise answers."

    graph = add_document_to_graph("doc1", doc1, graph)
    graph = add_document_to_graph("doc2", doc2, graph)
    
    # Add semantic edges
    add_semantic_edges(graph, threshold=0.8)
    # add_semantic_edges(graph, threshold=0.7)  # Use a lower threshold to increase graph connectivity

    
    # Query the graph
    query = "What is Graph RAG?"
    relevant_chunks = retrieve_relevant_chunks(graph, query)
    print(relevant_chunks)
    
    # Use Gemini for answering
    relevant_texts = [chunk for _, chunk in relevant_chunks]
    answer = query_with_gemini(relevant_texts, query)
    print("Answer from Gemini:", answer)

[]
Answer from Gemini: Not found.
